<a href="https://colab.research.google.com/github/divyam9876/Financial_Assistant/blob/main/Real_Time_Financial_PDF_Question_Answering_System_(RAG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faiss-cpu sentence-transformers pdfplumber

In [2]:
import numpy as np
import faiss
import pdfplumber
from sentence_transformers import SentenceTransformer

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from google.colab import files
uploaded = files.upload()

Saving financial_report.pdf to financial_report (2).pdf


In [5]:
pdf_path = list(uploaded.keys())[0]
print(pdf_path)


financial_report (2).pdf


In [6]:
def extract_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

pdf_text = extract_text(pdf_path)
print(pdf_text[:500])

ABC Technologies Inc.
Q3 FY2025 Earnings & Risk Disclosure Report
Executive Summary
ABC Technologies Inc. reported a 12% year-over-year revenue growth driven by cloud services
and enterprise software.
Revenue Breakdown
Cloud Services: 18% growth
Enterprise Software: 9% growth
International Revenue: 35%
Market Conditions
Markets experienced volatility due to rising interest rates, inflation, and currency fluctuations.
Risk Factors
Market Risk, Foreign Exchange Risk, Operational Cost Risk, Regulat


In [7]:
import re
pdf_lines= [line.strip() for line in pdf_text.split('\n') if line.split()]
company_name = pdf_lines[0]
# fy_matches = re.findall(r'FY\d{4}', pdf_text)
# financial_year = fy_matches[0] if fy_matches else 'Unknown'
# quarter_matches = re.findall(r'Q[1-4]', pdf_text, flags=re.IGNORECASE)
# quarter = quarter_matches[0] if quarter_matches else 'Unknown'
# print(quarter_matches)
report_name = pdf_lines[1] if len(pdf_lines) > 1 else "Unknown Report"
metadata_chunk = f'Company & Quarter & Financial Year : {company_name} {report_name}'
print('Metadata chunk:', metadata_chunk)

Metadata chunk: Company & Quarter & Financial Year : ABC Technologies Inc. Q3 FY2025 Earnings & Risk Disclosure Report


In [8]:
from importlib import metadata
import re

sections = re.split(r'(Executive Summary|Revenue Breakdown|Market Condition|Risk Factors|Liquidity|Forward Looking Statements)',pdf_text)

chunks=[]
for i in range(1,len(sections),2):
  header = sections[i].strip()
  content = sections[i+1].strip() if i+1<len(sections) else ""
  sentences=re.split(r'(?<=[.!?]) +', content)
  for sentence in sentences:
    sentence=sentence.strip()
    if sentence:
      chunk = header + ': ' + content
      chunks.append(chunk)

chunks = [metadata_chunk]+list(dict.fromkeys(chunks))
print(f'Total unique chunks: {len(chunks)}')
print('Example Chunk: ')
print(chunks)

Total unique chunks: 7
Example Chunk: 
['Company & Quarter & Financial Year : ABC Technologies Inc. Q3 FY2025 Earnings & Risk Disclosure Report', 'Executive Summary: ABC Technologies Inc. reported a 12% year-over-year revenue growth driven by cloud services\nand enterprise software.', 'Revenue Breakdown: Cloud Services: 18% growth\nEnterprise Software: 9% growth\nInternational Revenue: 35%', 'Market Condition: s\nMarkets experienced volatility due to rising interest rates, inflation, and currency fluctuations.', 'Risk Factors: Market Risk, Foreign Exchange Risk, Operational Cost Risk, Regulatory Risk.', 'Liquidity: Strong cash reserves with no major debt maturities in the next 12 months.', 'Forward Looking Statements: Management expects steady cloud demand while remaining cautious due to macroeconomic\nuncertainty.']


In [9]:
embeddings = model.encode(chunks).astype("float32")
print(embeddings)

[[-0.0066957   0.05515675 -0.0531394  ... -0.07945235  0.06707536
   0.03317456]
 [ 0.00448256  0.00782499 -0.01476602 ... -0.09745618  0.08260925
   0.01495416]
 [ 0.03551205 -0.05092022  0.01218896 ... -0.07115822  0.04261414
  -0.01157757]
 ...
 [ 0.08998279 -0.05628828 -0.06106069 ... -0.0327665   0.03967893
   0.01525182]
 [-0.02363469 -0.03215724 -0.05681185 ... -0.15217777  0.01335536
   0.02059145]
 [ 0.03761609 -0.01698037  0.03786248 ... -0.05048532 -0.05448167
  -0.00553021]]


In [10]:
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print(index)

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7a6837667d80> >


In [11]:
query = 'What risks are mentioned in the report?'
query_embedding = model.encode([query]).astype('float32')
_,indices = index.search(query_embedding,k=3)
context = '\n'.join([chunks[i] for i in indices[0]])
print(context)

Risk Factors: Market Risk, Foreign Exchange Risk, Operational Cost Risk, Regulatory Risk.
Company & Quarter & Financial Year : ABC Technologies Inc. Q3 FY2025 Earnings & Risk Disclosure Report
Forward Looking Statements: Management expects steady cloud demand while remaining cautious due to macroeconomic
uncertainty.


In [12]:
query = 'What is the name of the company?'
query_embedding = model.encode([query]).astype('float32')
_,indices = index.search(query_embedding,k=3)
context = '\n'.join([chunks[i] for i in indices[0]])
print(context)

Company & Quarter & Financial Year : ABC Technologies Inc. Q3 FY2025 Earnings & Risk Disclosure Report
Executive Summary: ABC Technologies Inc. reported a 12% year-over-year revenue growth driven by cloud services
and enterprise software.
Revenue Breakdown: Cloud Services: 18% growth
Enterprise Software: 9% growth
International Revenue: 35%


In [13]:
query = 'which financial year report is this?'
query_embedding = model.encode([query]).astype('float32')
_,indices = index.search(query_embedding,k=1)
context = '\n'.join([chunks[i] for i in indices[0]])
print(context)

Company & Quarter & Financial Year : ABC Technologies Inc. Q3 FY2025 Earnings & Risk Disclosure Report


In [14]:
query = 'which quarter is the report belongs to?'
query_embedding = model.encode([query]).astype('float32')
_,indices = index.search(query_embedding,k=3)
context = '\n'.join([chunks[i] for i in indices[0]])
print(context)

Company & Quarter & Financial Year : ABC Technologies Inc. Q3 FY2025 Earnings & Risk Disclosure Report
Executive Summary: ABC Technologies Inc. reported a 12% year-over-year revenue growth driven by cloud services
and enterprise software.
Liquidity: Strong cash reserves with no major debt maturities in the next 12 months.


In [15]:
query = 'Which report is this?'
query_embedding = model.encode([query]).astype('float32')
_,indices = index.search(query_embedding,k=3)
context = '\n'.join([chunks[i] for i in indices[0]])
print(context)

Company & Quarter & Financial Year : ABC Technologies Inc. Q3 FY2025 Earnings & Risk Disclosure Report
Executive Summary: ABC Technologies Inc. reported a 12% year-over-year revenue growth driven by cloud services
and enterprise software.
Revenue Breakdown: Cloud Services: 18% growth
Enterprise Software: 9% growth
International Revenue: 35%


In [16]:
query = 'What is the revenue breakdown?'
query_embedding = model.encode([query]).astype('float32')
_,indices = index.search(query_embedding,k=3)
context = '\n'.join([chunks[i] for i in indices[0]])
print(context)

Revenue Breakdown: Cloud Services: 18% growth
Enterprise Software: 9% growth
International Revenue: 35%
Executive Summary: ABC Technologies Inc. reported a 12% year-over-year revenue growth driven by cloud services
and enterprise software.
Company & Quarter & Financial Year : ABC Technologies Inc. Q3 FY2025 Earnings & Risk Disclosure Report


In [17]:
from transformers import pipeline

qa = pipeline("text2text-generation", model="google/flan-t5-base")

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}
"""

answer = qa(prompt, max_length=50)
print("Answer:", answer[0]['generated_text'])


Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: Cloud Services: 18% growth Enterprise Software: 9% growth International Revenue: 35%
